In [ ]:
# pip install pyspark

In [ ]:
from pyspark.sql import SparkSession
import pyspark as ps

In [ ]:
#abrindo sessão
spark=(SparkSession
       .builder
       .appName("Python Spark SQL basic example") 
        .config("spark.some.config.option", "some-value") 
        .getOrCreate())
conf = ps.SparkConf().setMaster("yarn-client").setAppName("sparK-mer")
conf.set("spark.executor.heartbeatInterval","3600s")

Leitura e conversão

In [ ]:
#leitura
pokemon_spark = (spark.read.format("csv")
      .option("header","true")
      .load("Dataset - Pokemon.csv"))
pokemon_spark

In [ ]:
pokemon_spark.printSchema()

In [ ]:
pokemon_spark.show(10)

In [ ]:
#spark -> pandas
print(type(pokemon_spark))
pokemon_pd = pokemon_spark.toPandas()
print(type(pokemon_pd))
pokemon_pd.head(2)

In [ ]:
#pandas -> spark
pokemon_spark_2=spark.createDataFrame(pokemon_pd)
print(type(pokemon_spark_2))


Operações

In [ ]:
pokemon_spark.show()

In [ ]:
#drop
pokemon_spark.drop('_c0','egg_type_number','type_number').show(15)

In [ ]:
#select
pokemon_spark.select('name','species','type_1','type_2','attack','defense','growth_rate').show(8)

In [ ]:
#rename
pokemon_spark.select('name','species','type_1','type_2','ability_1',
                     'ability_hidden','growth_rate').withColumnRenamed('name', 'nome').withColumnRenamed('ability_1', 'ability').show(11)

In [ ]:
#criando coluna com valor fixo
import pyspark.sql.functions as F
from pyspark.sql.functions import col, lit, when

pokemon_spark.select('name','species','type_1',
                     'type_2','growth_rate').withColumn('max_generation', lit('7ª')).show(9)

In [ ]:
#criando coluna com 'dependencia'
pokemon_spark.select('name','species','attack',
                     'defense','growth_rate').withColumn('attack/defense',F.round(col('attack')/col('defense'),1)).show(12)

In [ ]:
#criando coluna concatenada
pokemon_spark.select('name','species','type_1',
                     'type_2','growth_rate').withColumn('type', F.concat(F.col('type_1'), F.lit(' - '), F.col('type_2'))).show(8)


In [ ]:
#criando coluna condicional
pokemon_filtro=pokemon_spark.select('name','species','attack',
                     'defense','growth_rate').withColumn('attack/defense',F.round(col('attack')/col('defense'),1))
print(pokemon_filtro.show(5))
pokemon_filtro.withColumn('forte',
                                        when(col('attack/defense')>1,'ataque')
                                       .when(col('attack/defense')<1,'defesa')
                                       .otherwise('equilibrado')).show(5)

In [ ]:
#filtro
pokemon_spark.select('name','species','type_1',
                     'type_2','growth_rate').filter(col('type_1')=='Ghost').show(10)

In [ ]:
pokemon_spark.select('name','species','attack',
                     'defense','growth_rate').filter(col('attack')>180).show(10)

In [ ]:
pokemon_spark.select('name','species','attack',
                     'defense','growth_rate').filter("attack>180").show(10)

In [ ]:
pokemon_spark.select('name','species','type_1',
                     'type_2','growth_rate','attack').filter((col('type_1')=='Ghost')&(col('attack')<50)).show(10)

In [ ]:
#contagem
pokemon_spark.select('name','species','type_1',
                     'type_2','growth_rate','attack').filter((col('type_1')=='Ghost')&(col('attack')<50)).count()

In [ ]:
spark.stop() #parada do ambiente spark